In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib as mpl
import matplotlib.pyplot as plt
import sys
import os

# System definitions

In [ ]:
from nsflows.systems.lennard_jones import lennard_jones
from nsflows.systems.uniforms import box_uniform
from nsflows.tools.util import density_from_box_length

n_particles = 8
dimensions = 2
cutin = 0.8

# State points provided with this repository, keyed by the density as reported in
# the paper. The box length is the exact value the simulations were run at, and is
# what names the data directories under data/lj/; the key is that density rounded
# to two decimals. Change `density` to switch between state points: everything
# below, including which initial samples are loaded, follows from it.
box_length_for_density = {
    0.95: 2.9,
    0.73: 3.3,
}

density = 0.95

if density not in box_length_for_density:
    raise KeyError(
        f"No data provided for density={density}; "
        f"available: {sorted(box_length_for_density)}"
    )

box_length = box_length_for_density[density]
rho = density_from_box_length(box_length, n_particles, dimensions)

print(f"Selected density {density}: box length {box_length}, exact density {rho:.6f}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

box_uniform_2D = box_uniform(n_particles=n_particles, dimensions=dimensions, device=device, box_length=box_length)
LJ_disks = lennard_jones(n_particles=n_particles, dimensions=dimensions, rho=rho, device=device, cutin=cutin, lrc=True)

## Define Parameters

In [ ]:
# Load Previous Training
load = False

# Nested Sampling Parameters
live_samples = 10000

# The initial live-sample set is picked from data/lj/ according to live_samples and
# the state point selected above.
init_samples_dir = "../data/lj"
init_samples_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", f"L{box_length}", "samples_init.pt")
if not os.path.exists(init_samples_filepath):
    raise FileNotFoundError(
        f"No initial samples shipped for live_samples={live_samples} at box length "
        f"{box_length} (looked for {init_samples_filepath})"
    )

max_ns_iterations = 500000
n_propagate = 1000
update_step = True
turn_on_nf = -1   # negative disables the flow: standard nested sampling


# Output Folder Definition

In [ ]:
from nsflows.tools.util import generate_unique_identifier, remove_empty_directories, generate_output_directory

root_folder = "./output/L2.9/"
remove_empty_directories(root_folder=root_folder)

if load:
    output_dir = "./cluster_run/L2.9/FT"
    print(f"Run Folder: {output_dir}")
else:
    run_id = generate_unique_identifier()
    output_dir = generate_output_directory(run_id, root_folder=root_folder)

In [ ]:
# ============================================================
# Generate summary for new runs
# OR
# Read and display existing summary for loaded runs
# ============================================================

from pathlib import Path
from datetime import datetime
import json

summary_txt_path = Path(output_dir) / "simulation_summary.txt"
summary_json_path = Path(output_dir) / "simulation_summary.json"

if not load:

    # --------------------------------------------------------
    # Build parameter summary dictionary
    # --------------------------------------------------------

    summary = {
        "timestamp": datetime.now().isoformat(),

        "system": {
            "n_particles": n_particles,
            "dimensions": dimensions,
            "box_length": box_length,
            "cutin": cutin,
            "rho": rho,
            "device": str(device),
        },

        "nested_sampling": {
            "init_samples_filepath": init_samples_filepath,
            "live_samples": live_samples,
            "max_ns_iterations": max_ns_iterations,
            "n_propagate": n_propagate,
            "update_step": update_step,
            "turn_on_nf": turn_on_nf,
        },

    }

    # --------------------------------------------------------
    # Ensure output directory exists
    # --------------------------------------------------------

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Save JSON summary
    # --------------------------------------------------------

    with open(summary_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    # --------------------------------------------------------
    # Save readable text summary
    # --------------------------------------------------------

    with open(summary_txt_path, "w") as f:

        f.write("====================================================\n")
        f.write("Simulation Summary\n")
        f.write("====================================================\n\n")

        f.write(f"Generated: {summary['timestamp']}\n\n")

        f.write("SYSTEM PARAMETERS\n")
        f.write("-----------------\n")
        for k, v in summary["system"].items():
            f.write(f"{k}: {v}\n")

        f.write("\nNESTED SAMPLING PARAMETERS\n")
        f.write("--------------------------\n")
        for k, v in summary["nested_sampling"].items():
            f.write(f"{k}: {v}\n")

    print(f"Summary files written to:\n")
    print(f"  JSON : {summary_json_path}")
    print(f"  TEXT : {summary_txt_path}")

else:

    # --------------------------------------------------------
    # Read and print existing summary
    # --------------------------------------------------------

    try:
        with open(summary_txt_path, "r") as f:
            summary_contents = f.read()

        print("====================================================")
        print("Loaded Run Summary")
        print("====================================================\n")

        print(summary_contents)

    except FileNotFoundError:
        print("WARNING: No simulation summary file found.")
        print(f"Expected location:\n{summary_txt_path}")

    except Exception as e:
        print("ERROR while reading simulation summary:")
        print(e)

# Nested Sampling

In [ ]:
import os

from nsflows.samplers.monte_carlo import rejection_monte_carlo
from nsflows.nested_sampling import nested_sampling

rejection_sampler = rejection_monte_carlo(system=LJ_disks, n_cycles=100, step_size=1.2, transform=True)

if load:
    acceptance, umax_plt = np.loadtxt(os.path.join(output_dir, "output.txt"), usecols=(2,3), unpack=True)
    samples = torch.load(os.path.join(output_dir, "samples.pt"))
    U_samples = umax_plt[-1]

else:
    samples, U_samples, acceptance, umax_plt = nested_sampling(K=live_samples,
                                                            system=LJ_disks,
                                                            std_propagator=rejection_sampler,
                                                            init_samples_filepath=init_samples_filepath,
                                                            max_iters=max_ns_iterations,
                                                            n_propagate=n_propagate,
                                                            update_step=update_step,
                                                            turn_on_nf=turn_on_nf,
                                                            iprint=1,
                                                            isavesamp=500,
                                                            outputdir=output_dir,
                                                            disable_pbar=True)


# Plot Output

## Live Sets

In [ ]:
# Helpers for comparing configurations up to the symmetries of the system: a pi/2
# rotation of the box and a relabelling of the particles. They live in
# nsflows.tools.util so that the notebooks and the analysis scripts share one
# implementation.
from nsflows.tools.util import (
    remove_outermost_particle,
    align_config,
)


In [ ]:
# Live set, plotted roughly every `plot_every_iters` nested sampling iterations. The
# run saves samples_<iter>.pt / U_max_<iter>.pt every `isavesamp` iterations, so the
# spacing is rounded up to whatever snapshots exist.
plot_every_iters = 100000

# Align every configuration to a common reference, up to a pi/2 rotation of the box
# and a relabelling of the particles, so the snapshots can be compared directly.
# Set align = False to plot the raw configurations instead.
align = True

if align:
    ref_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", f"L{box_length}", "samples_ref.pt")
    if not os.path.exists(ref_filepath):
        raise FileNotFoundError(
            f"No reference configuration shipped for box length {box_length} "
            f"(looked for {ref_filepath}); set align = False to skip the alignment"
        )
    reff_config = torch.load(ref_filepath, map_location=device)
    refff_config = remove_outermost_particle(reff_config.view(-1, LJ_disks.n_particles, LJ_disks.dimensions))
    ref_config = refff_config[0].clone().unsqueeze(0)

snapshots = sorted(f for f in os.listdir(output_dir)
                   if f.startswith("samples_") and f.endswith(".pt"))
if not snapshots:
    print(f"No sample snapshots found in {output_dir}")

# The snapshots actually shown, reused by the radial distribution functions below.
selected_snapshots = []
last_plotted = None
for filename in snapshots:
    iteration = int(filename[len("samples_"):-len(".pt")])
    if last_plotted is not None and iteration - last_plotted < plot_every_iters:
        continue
    last_plotted = iteration
    selected_snapshots.append(filename)

for filename in selected_snapshots:
    iteration = int(filename[len("samples_"):-len(".pt")])

    live_set = torch.load(os.path.join(output_dir, filename), map_location=device)
    live_set = live_set.view(-1, LJ_disks.n_particles, LJ_disks.dimensions)

    if align:
        live_set = align_config(live_set, ref_config, n_particles, dimensions, box_length)

    live_set = live_set.cpu().numpy()

    fig_size = (10 * 0.393701, 10 * 0.393701)
    fig, ax = plt.subplots(figsize = fig_size, dpi = 400, tight_layout=True)

    ax.scatter(live_set[:,:,0], live_set[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C0")
    ax.set_xlim(-box_length/2, box_length/2)
    ax.set_ylim(-box_length/2, box_length/2)
    ax.set_xticks([-box_length/2, 0, box_length/2])
    ax.set_yticks([-box_length/2, 0, box_length/2])
    ax.set_title(f"Iteration {iteration}")

    plt.show()


## Radial Distribution Functions

In [ ]:
from nsflows.tools.observables import rdf

# Radial distribution function of the same live sets plotted above.
gofrs = []
for filename in selected_snapshots:
    live_set = torch.load(os.path.join(output_dir, filename), map_location=device)
    r, gofr = rdf(live_set, n_particles=LJ_disks.n_particles, dimensions=LJ_disks.dimensions, box_length=LJ_disks.box_length)
    gofrs.append(gofr)

fig_size = (10 * 0.393701, 7.5 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400)

for filename, gofr in zip(selected_snapshots, gofrs):
    iteration = int(filename[len("samples_"):-len(".pt")])
    plt.plot(r, gofr, label=f"{iteration}")

plt.axhline(1, color="k", ls=":")
plt.xlabel(r"$r$")
plt.ylabel(r"$g(r)$")
plt.legend(title="Iteration", frameon=False, fontsize="x-small")
plt.savefig(os.path.join(output_dir, f"rdfs.png"))
plt.show()


## Energy vs. Iteration and Density of States vs. Iteration

In [ ]:
vals, bins = np.histogram(umax_plt-umax_plt.min(), bins=200)
bin_centers = .5*(bins[1:] + bins[:-1])

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# umax_plt covers the iterations that actually ran, which is fewer than
# max_ns_iterations if the run was stopped early.
iters = np.arange(len(umax_plt))/1e4

# Horizontal guides at a fixed number of evenly spaced energy levels.
n_guide_lines = 50
guide_step = max(1, len(umax_plt)//n_guide_lines)

fig_size = (10 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(1, 2, figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

ax[0].plot(iters, (umax_plt-umax_plt.min()), color="C0")
for u in ((umax_plt[guide_step//2::guide_step]-umax_plt.min())):

    ax[0].axhline(u, ls=":", lw=0.5, color="k")

ax[0].set_ylabel("Energy")
ax[0].set_xlabel(r"Iteration ($\times 10^4$)")
ax[0].set_xticks([0, iters[-1]])

ax_inset = inset_axes(ax[0], width="50%", height="30%", loc='upper right', borderpad=.5)
ax_inset.plot(iters, (umax_plt-umax_plt.min()), color="C0")
for u in ((umax_plt[guide_step//2::guide_step]-umax_plt.min())):

    ax_inset.axhline(u, ls=":", lw=0.5, color="k")
ax_inset.set_yscale("log")
ax_inset.set_ylim(4.0e-2,1.e3)

ax[1].plot(vals, bin_centers, color="C1")
ax[1].set_xlabel(r"Samples")
ax[1].set_xscale('log')

plt.savefig(os.path.join(output_dir, f"evsiter.png"))
plt.show()

## Timings

In [ ]:
# Elapsed time for this run, read from the timings file written by nested_sampling.
timings_filepath = os.path.join(output_dir, "timings.txt")

with open(timings_filepath) as f:
    notes = [line.lstrip("#").strip() for line in f if line.startswith("#")]

for key in ("Run started", "Run ended", "Elapsed time", "Run interrupted"):
    for note in notes:
        if note.startswith(key):
            print(note)

# Without the flow, STD_NS is the only non-zero column: the time spent sampling.
sampling_time = np.genfromtxt(timings_filepath, skip_header=2, names=True, comments="#")["STD_NS"].sum()
n_iters = len(umax_plt) - 1

print(f"Iterations completed: {n_iters}")
if n_iters > 0:
    print(f"Sampling time: {sampling_time:.1f} s ({1e3*sampling_time/n_iters:.2f} ms per iteration)")
